# 🌳 Almond Sentinel: Stress Detection (6 classes)

**Goal:** for each tree and each day, detect which stress the almond tree is under, using only the node's sensors (soil moisture, soil temperature, air temperature and humidity, light, piezo) plus farm information.

| Class (`primary_stress`) | Meaning |
|---|---|
| `none` | no stress |
| `water_deficit` | soil too dry for the tree |
| `waterlogging` | root zone saturated too long (no oxygen for the roots) |
| `heat` | hot days (≥ 35 / 38 / 40 °C) while the tree is in leaf |
| `frost` | freezing temperature at bloom or small-nut stage |
| `rootzone` | the tree is stressed **although the soil is moist** (salinity, root damage) |

**Dataset:** `daily_features.csv`, one row per tree per day: 216 trees, 6 soil types, 2 sites, 3 seasons (2023-2025).

**Plan**
1. Load the data from Google Drive
2. Quick exploration
3. Data preparation: leakage columns, missing values (None), duplicates, impossible values, outliers
4. Feature selection and encoding
5. Train/test split **by season and farm** (train: 2023 + 2024 farms, test: unseen 2025 farms)
6. Train and compare 9 algorithms with cross-validation (metric: **macro-F1**)
7. Tune the best algorithm
8. Final evaluation on the unseen farms: per class, per soil type, important features
9. Save the model and predict new days

> **How to run:** in Colab, *Runtime → Run all* (about 15–20 minutes; model comparison and tuning take most of the time). When asked, allow access to Google Drive.
> If the dataset folder was *shared with you*, first right-click it in Drive → **Organize → Add shortcut → My Drive**.

## 0. Setup

In [ ]:
import os, sys, glob, time, warnings, importlib, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 170)
sns.set_theme(style="whitegrid")

for pkg in ["xgboost", "lightgbm"]:            # pre-installed on Colab; install if missing
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GroupKFold, cross_validate, RandomizedSearchCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (f1_score, balanced_accuracy_score, accuracy_score, classification_report,
                             confusion_matrix)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
from IPython.display import display
print("Libraries ready")

## 1. Load the data from Google Drive

In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

REL_PATH = "AOTN TSYP14 Challenge - IEEE ESPRIT SB/used data/stress/daily_features.csv"
candidates = [
    f"/content/drive/MyDrive/{REL_PATH}",       # folder in My Drive (or a shortcut to it)
    f"/content/drive/Shareddrives/{REL_PATH}",  # folder is a Shared drive
    "daily_features.csv",                       # file uploaded next to the notebook
]
DATA_PATH = next((p for p in candidates if os.path.exists(p)), None)
if DATA_PATH is None and IN_COLAB:
    print("Not found at the expected path, searching Drive (can take a minute)...")
    hits = [h for h in glob.glob("/content/drive/MyDrive/**/daily_features.csv", recursive=True) +
            glob.glob("/content/drive/Shareddrives/**/daily_features.csv", recursive=True) if "stress" in h]
    DATA_PATH = hits[0] if hits else None
assert DATA_PATH is not None, ("daily_features.csv not found. Check the folder name, add a shortcut of the "
                               "shared folder to My Drive, or upload the file to Colab.")
print("Using:", DATA_PATH)

raw = pd.read_csv(DATA_PATH, parse_dates=["date"], low_memory=False)
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head(3)

## 2. Quick exploration

In [ ]:
TARGET = "primary_stress"
counts = raw[TARGET].value_counts()
print(pd.DataFrame({"days": counts, "share": (counts / len(raw)).round(3)}))
print("\nThe classes are very imbalanced (frost is 0.3 %), so we judge models with macro-F1: "
      "the average F1 over the 6 classes, where every class counts the same.")

fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
counts.plot.bar(ax=ax[0], color="#4C72B0"); ax[0].set_title("Days per stress class")
(pd.crosstab(raw["soil_type"], raw[TARGET], normalize="index") * 100)\
    .reindex(["sand", "loamy_sand", "sandy_loam", "loam", "clay_loam", "clay"])\
    .plot.bar(stacked=True, ax=ax[1], colormap="tab10")
ax[1].set_title("Stress mix by soil type (%)"); ax[1].legend(bbox_to_anchor=(1, 1), fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# A few key features by class
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for a, col in zip(ax, ["rew_predawn", "hours_near_saturation", "tair_max", "uptake_per_etc_7d"]):
    sns.boxplot(data=raw, x=TARGET, y=col, ax=a, showfliers=False)
    a.set_title(col); a.tick_params(axis="x", rotation=45); a.set_xlabel("")
plt.tight_layout(); plt.show()

## 3. Data preparation

### 3.1 Remove leakage columns
These columns are **answers or hidden truth**, so the model must not see them:
* other label columns: `any_stress`, `max_severity`, `sev_*` (they are built from the same truth as the target)
* `gt_*` columns (hidden simulator truth: real stem water potential, salinity, events...) and `scenario`
* identifiers `tree_id`, `site`. We keep `orchard_id`, `soil_type` and `date` aside: they are used for the split, the grouping and the per-soil analysis, but **not as features**.

In [ ]:
LABEL_COLS = ["any_stress", "max_severity", "sev_water_deficit", "sev_waterlogging", "sev_heat", "sev_frost", "sev_rootzone"]
GT_COLS = [c for c in raw.columns if c.startswith("gt_")] + ["scenario"]
df = raw.drop(columns=LABEL_COLS + GT_COLS + ["tree_id", "site"]).copy()
KEEP_ASIDE = ["orchard_id", "soil_type", "date"]
print(f"Removed {len(LABEL_COLS + GT_COLS) + 2} columns -> {df.shape[1]} left (incl. target and {len(KEEP_ASIDE)} helper columns)")

### 3.2 Missing values (None / NaN)
⚠️ **`df.dropna()` would delete 93 % of the rows.** Most empty cells are *empty on purpose*:

| Kind | Columns | Meaning | Action |
|---|---|---|---|
| Empty on purpose | `drawdown_*`, `uptake_*` (+ their 7-day and orchard versions) | water use cannot be measured on rain/irrigation days or in winter | median filled **inside the model pipeline** (learned on training data only) + flag `uptake_measured` |
| Empty on purpose | `irrigation_water_ec_dsm` | rainfed farm: no irrigation water | 0 |
| Unknown | `irrigation_water_ec_dsm` on drip farms | farmer never measured the water salinity | median (pipeline) + flag `ec_unknown` |
| No history yet | `piezo_z14`, `vwc_predawn_trend_3d`, `vwc_predawn_delta_1d` | first days of a series | 0 (= no change) |
| External data gap | `rain_api_mm` | weather station gap | 0 + flag `rain_api_missing` |
| **Real missing** | sensor readings (soil, air, piezo...) | node outage | **remove the row** |

In [ ]:
UPTAKE_COLS = ["drawdown_day_mm", "drawdown_per_etc", "uptake_est_mm", "uptake_per_etc", "drawdown_per_etc_7d",
               "uptake_per_etc_7d", "drawdown_per_etc_7d_vs_orchard", "uptake_per_etc_7d_vs_orchard"]
NO_HISTORY_COLS = ["piezo_z14", "vwc_predawn_trend_3d", "vwc_predawn_delta_1d"]

def add_missing_flags_and_rule_fills(x):
    # Deterministic rules (no statistics), used for training AND for new data
    x = x.copy()
    x["uptake_measured"] = x["uptake_per_etc"].notna().astype(int)
    x["uptake_7d_measured"] = x["uptake_per_etc_7d"].notna().astype(int)
    rainfed = x["irrigation_system"] != "drip"
    x.loc[rainfed, "irrigation_water_ec_dsm"] = 0.0
    x["ec_unknown"] = x["irrigation_water_ec_dsm"].isna().astype(int)
    x["rain_api_missing"] = x["rain_api_mm"].isna().astype(int)
    x["rain_api_mm"] = x["rain_api_mm"].fillna(0.0)
    x[NO_HISTORY_COLS] = x[NO_HISTORY_COLS].fillna(0.0)
    x["leaf_on"] = x["leaf_on"].astype(str).str.lower().isin(["true", "1"]).astype(int)
    return x

before = len(df)
print(f"Rows with at least one empty cell BEFORE handling: {df.isna().any(axis=1).mean():.1%}")
df = add_missing_flags_and_rule_fills(df)

STAT_FILL = UPTAKE_COLS + ["irrigation_water_ec_dsm"]          # filled later by the pipeline (train medians)
real_missing = df.drop(columns=STAT_FILL).isna().any(axis=1)
print("Real missing values per column (top 10):")
print(df.drop(columns=STAT_FILL).isna().sum().sort_values(ascending=False).head(10).to_string())
print("\nClass mix of the rows with real missing values:")
print(df.loc[real_missing, TARGET].value_counts(normalize=True).round(3).to_string())
df = df[~real_missing].reset_index(drop=True)
print(f"\nRemoved {before - len(df):,} rows with sensor outages ({(before - len(df)) / before:.1%}); "
      f"{len(df):,} rows left. The class mix of removed rows matches the whole dataset -> no bias.")

### 3.3 Duplicates

In [ ]:
dups = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicate rows removed: {dups}")

### 3.4 Impossible values (sensor errors)
True outliers: values the sensors cannot produce, or known error codes (DS18B20: **85 °C** at power-on, **−127 °C** when disconnected). These rows are removed.

In [ ]:
VALID_RANGES = {
    "tair_max": (-40, 60), "tair_min": (-40, 60), "tair_mean": (-40, 60),
    "rh_min": (0, 100), "rh_max": (0, 100), "rh_mean": (0, 100),
    "tsoil_min": (-20, 60), "tsoil_max": (-20, 60), "tsoil_mean": (-20, 60),
    "vwc_min": (0, 0.6), "vwc_max": (0, 0.6), "vwc_mean": (0, 0.6),
    "light_mean_adc": (0, 4095), "piezo_rate_midday": (0, 500), "piezo_rms_midday": (0, 100),
    "n_valid_hours": (1, 24),
}
bad = pd.Series(False, index=df.index)
for col, (lo, hi) in VALID_RANGES.items():
    bad |= ~df[col].between(lo, hi)
for col in ["tsoil_min", "tsoil_max", "tsoil_mean"]:
    bad |= df[col].isin([85.0, -127.0])
print(f"Rows with impossible values removed: {bad.sum()}")
df = df[~bad].reset_index(drop=True)
print(f"Rows left: {len(df):,}")

### 3.5 Statistical outliers (IQR rule)
Before deleting anything, let's see **which days** the IQR rule would flag:

In [ ]:
CATEGORICAL = ["irrigation_system", "pheno_stage"]
numeric_cols = [c for c in df.columns if c not in CATEGORICAL + KEEP_ASIDE + [TARGET] and pd.api.types.is_numeric_dtype(df[c])]
CONTINUOUS = [c for c in numeric_cols if df[c].nunique() > 15]

def iqr_fences(data, cols, k):
    # Fences only for columns with a real spread: count features that are 0 on most days
    # (e.g. hours_above_35) have IQR = 0, and every non-zero value (= every hot day!) would be "an outlier"
    q1, q3 = data[cols].quantile(0.25), data[cols].quantile(0.75)
    iqr = q3 - q1
    keep = iqr[iqr > 0].index
    return q1[keep] - k * iqr[keep], q3[keep] + k * iqr[keep]

q1, q3 = df[CONTINUOUS].quantile(0.25), df[CONTINUOUS].quantile(0.75)
print("Columns skipped (IQR = 0, zero on most days):", list((q3 - q1)[(q3 - q1) == 0].index), "\n")
stress_rate = (df[TARGET] != "none").mean()
for k in [1.5, 3.0]:
    lo, hi = iqr_fences(df, CONTINUOUS, k)
    flagged = ((df[lo.index] < lo) | (df[hi.index] > hi)).any(axis=1)
    print(f"k = {k}: {flagged.mean():.1%} of rows flagged | stressed days among them: "
          f"{(df.loc[flagged, TARGET] != 'none').mean():.1%} (overall {stress_rate:.1%})")
    print("   classes that would lose the most days:",
          (df.loc[flagged, TARGET].value_counts() / df[TARGET].value_counts()).dropna().sort_values(ascending=False)
          .head(3).round(2).to_dict())

**Result:** the flagged days are mostly **stress days**, for example waterlogged soil or heat waves. These are exactly what we want to detect, not sensor errors.
Deleting them would remove most of the rare classes.

**Strategy used:**
* impossible values → **removed** (3.4)
* statistical extremes → **capped** at the k = 3 IQR fences. The capping step sits **inside the model pipeline**, so the fences are learned from training data only, both in cross-validation and in the final model. Extreme values can no longer dominate distance-based models (KNN, logistic regression, MLP), and no stress day is lost.

To delete them anyway, set `OUTLIER_STRATEGY = "remove"`. Rows are then removed from the training set only; the test farms stay untouched.

In [ ]:
OUTLIER_STRATEGY = "cap"   # "cap" (recommended) or "remove"
IQR_K = 3.0

class IQRCapper(BaseEstimator, TransformerMixin):
    # Clip continuous columns to [Q1 - k*IQR, Q3 + k*IQR] learned on the training data
    def __init__(self, columns=None, k=3.0):
        self.columns, self.k = columns, k
    def fit(self, X, y=None):
        cols = [c for c in (self.columns or []) if c in X.columns]
        q1, q3 = X[cols].quantile(0.25), X[cols].quantile(0.75)
        iqr = q3 - q1
        self.cols_ = list(iqr[iqr > 0].index)            # skip zero-IQR count columns
        self.low_ = q1[self.cols_] - self.k * iqr[self.cols_]
        self.high_ = q3[self.cols_] + self.k * iqr[self.cols_]
        return self
    def transform(self, X):
        X = X.copy()
        X[self.cols_] = X[self.cols_].clip(self.low_, self.high_, axis=1)
        return X

## 4. Feature selection and encoding
* **Soil:** we do **not** give the model the soil name or raw soil-moisture values (`vwc_*`, `theta_*`). Tested on unseen soils, the model then learns shortcuts like "waterlogging = clay".
  Instead it gets soil-**normalised** moisture (`rew_*`, `sat_ratio_max`, `hours_near_saturation`...), which already uses each soil's field capacity and wilting point. This generalises to new soils.
* `irrigation_system`, `pheno_stage` → one-hot columns. `pheno_stage_id` → dropped (duplicate of `pheno_stage`).

In [ ]:
ABSOLUTE_SOIL_COLS = ["vwc_mean", "vwc_min", "vwc_max", "vwc_predawn", "vwc_midnight", "vwc_morning", "vwc_evening",
                      "theta_fc", "theta_wp", "theta_sat", "taw_mm_per_m", "drainage_class",
                      "vwc_predawn_vs_orchard", "vwc_predawn_delta_1d", "vwc_predawn_trend_3d"]
USE_SOIL_IDENTITY = False   # set True to compare: raw VWC + soil columns as features

def encode(x):
    x = pd.get_dummies(x, columns=CATEGORICAL, prefix=["irr", "stage"], dtype=int)
    return x.drop(columns=["pheno_stage_id"], errors="ignore")

data = encode(df)
drop_feats = KEEP_ASIDE + [TARGET] + ([] if USE_SOIL_IDENTITY else ABSOLUTE_SOIL_COLS)
FEATURES = [c for c in data.columns if c not in drop_feats]
CONTINUOUS = [c for c in CONTINUOUS if c in FEATURES]
print(f"{len(FEATURES)} features")
print(FEATURES)

## 5. Train/test split by season **and** farm
Each farm (orchard) appears in only one season, so:
* **train** = the 24 farms of 2023 and 2024
* **test** = the 12 farms of 2025, never seen during training (new farms, new weather)

A random split would put neighbouring days of the same tree in both train and test, and the scores would look much better than reality.
Inside the training set, cross-validation keeps each farm in a single fold (`GroupKFold` grouped by `orchard_id`).

In [ ]:
is_test = data["orchard_id"].str.endswith("Y3")
train, test = data[~is_test].reset_index(drop=True), data[is_test].reset_index(drop=True)

if OUTLIER_STRATEGY == "remove":
    lo, hi = iqr_fences(train, CONTINUOUS, IQR_K)
    keep = ~((train[lo.index] < lo) | (train[hi.index] > hi)).any(axis=1)
    print(f"Removing {(~keep).sum()} outlier rows from the TRAINING set only")
    train = train[keep].reset_index(drop=True)

label_enc = LabelEncoder().fit(data[TARGET])
CLASSES = list(label_enc.classes_)
X_train, y_train = train[FEATURES].astype(float), label_enc.transform(train[TARGET])
X_test, y_test = test[FEATURES].astype(float), label_enc.transform(test[TARGET])
groups = train["orchard_id"].values
print(f"Train: {len(X_train):,} days from {train['orchard_id'].nunique()} farms | "
      f"Test: {len(X_test):,} days from {test['orchard_id'].nunique()} unseen farms")
print(pd.DataFrame({"train": train[TARGET].value_counts(), "test": test[TARGET].value_counts()}))

## 6. Train and compare algorithms
Every algorithm is wrapped in the same pipeline:
`IQR capping → median filling of the "empty on purpose" columns → (scaling, for LR / KNN / MLP) → model`.
All preparation steps are learned **inside** each training fold, so nothing leaks from validation data.

**Class imbalance:** `class_weight="balanced"` makes rare classes (frost, heat) count more. XGBoost has no `class_weight`, so we give it balanced *sample weights* instead.

**Metric: macro-F1.** F1 is computed for each of the 6 classes and averaged, so a model that ignores frost and heat is penalised.

In [ ]:
class BalancedXGBClassifier(XGBClassifier):
    # XGBoost + balanced class weights (like class_weight="balanced" in scikit-learn)
    def fit(self, X, y, **kwargs):
        kwargs.setdefault("sample_weight", compute_sample_weight("balanced", y))
        return super().fit(X, y, **kwargs)

def make_pipeline(model, scale=False):
    steps = [("cap", IQRCapper(CONTINUOUS, IQR_K)), ("impute", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scale", StandardScaler()))
    return Pipeline(steps + [("model", model)])

models = {
    "Logistic Regression": make_pipeline(LogisticRegression(max_iter=3000, class_weight="balanced"), scale=True),
    "KNN": make_pipeline(KNeighborsClassifier(n_neighbors=15, weights="distance"), scale=True),
    "Decision Tree": make_pipeline(DecisionTreeClassifier(max_depth=12, min_samples_leaf=20, class_weight="balanced",
                                                          random_state=RANDOM_STATE)),
    "Random Forest": make_pipeline(RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight="balanced_subsample",
                                                          n_jobs=-1, random_state=RANDOM_STATE)),
    "Extra Trees": make_pipeline(ExtraTreesClassifier(n_estimators=200, min_samples_leaf=3, class_weight="balanced",
                                                      n_jobs=-1, random_state=RANDOM_STATE)),
    "Hist Gradient Boosting": make_pipeline(HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08,
                                                                           class_weight="balanced", random_state=RANDOM_STATE)),
    "XGBoost": make_pipeline(BalancedXGBClassifier(n_estimators=300, learning_rate=0.08, max_depth=6, subsample=0.9,
                                                   colsample_bytree=0.7, tree_method="hist", n_jobs=-1,
                                                   random_state=RANDOM_STATE)),
    "LightGBM": make_pipeline(LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, subsample=0.9,
                                             subsample_freq=1, colsample_bytree=0.7, class_weight="balanced",
                                             verbose=-1, random_state=RANDOM_STATE)),
    "Neural Network (MLP)": make_pipeline(MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-3, max_iter=200,
                                                        early_stopping=True, random_state=RANDOM_STATE), scale=True),
}
SCORING = {"macro_f1": "f1_macro", "balanced_acc": "balanced_accuracy", "weighted_f1": "f1_weighted", "accuracy": "accuracy"}
cv = GroupKFold(n_splits=4)

results = []
for name, model in models.items():
    t0 = time.time()
    r = cross_validate(model, X_train, y_train, groups=groups, cv=cv, scoring=SCORING, n_jobs=1)
    results.append({"model": name, **{k: r[f"test_{k}"].mean() for k in SCORING},
                    "macro_f1_std": r["test_macro_f1"].std(), "fit_s": r["fit_time"].mean()})
    print(f"{name:<24} macro-F1 {r['test_macro_f1'].mean():.3f} ± {r['test_macro_f1'].std():.3f}   ({time.time() - t0:.0f}s)")

cv_results = pd.DataFrame(results).set_index("model").sort_values("macro_f1", ascending=False)
cv_results.round(3)

In [ ]:
ax = cv_results["macro_f1"].sort_values().plot.barh(figsize=(9, 5), color="#4C72B0",
                                                      xerr=cv_results["macro_f1_std"].reindex(cv_results["macro_f1"].sort_values().index))
ax.set_xlabel("Cross-validated macro-F1 (higher is better)"); ax.set_title("Model comparison (4 folds, grouped by farm)")
for i, v in enumerate(cv_results["macro_f1"].sort_values()):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center")
plt.tight_layout(); plt.show()

BEST_NAME = cv_results.index[0]
print("Best algorithm:", BEST_NAME)

## 7. Tune the best algorithm
Randomized search over the winner's main hyper-parameters (same farm-grouped cross-validation).

In [ ]:
PARAM_GRIDS = {   # compact grids so the search stays around 5 minutes on Colab
    "LightGBM": {"model__n_estimators": [200, 350], "model__learning_rate": [0.05, 0.1],
                 "model__num_leaves": [15, 31, 63], "model__min_child_samples": [10, 30],
                 "model__colsample_bytree": [0.5, 0.7], "model__reg_lambda": [0, 2]},
    "XGBoost": {"model__n_estimators": [200, 350], "model__learning_rate": [0.05, 0.1],
                "model__max_depth": [4, 6, 8], "model__min_child_weight": [1, 5],
                "model__colsample_bytree": [0.5, 0.7], "model__subsample": [0.8, 1.0]},
    "Hist Gradient Boosting": {"model__max_iter": [150, 300], "model__learning_rate": [0.05, 0.1],
                               "model__max_leaf_nodes": [15, 31, 63], "model__min_samples_leaf": [10, 30],
                               "model__l2_regularization": [0, 2]},
    "Random Forest": {"model__n_estimators": [150, 300], "model__max_depth": [None, 20],
                      "model__min_samples_leaf": [1, 3, 5], "model__max_features": ["sqrt", 0.25]},
    "Extra Trees": {"model__n_estimators": [150, 300], "model__max_depth": [None, 20],
                    "model__min_samples_leaf": [1, 3, 5], "model__max_features": ["sqrt", 0.25]},
    "Logistic Regression": {"model__C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3]},
    "KNN": {"model__n_neighbors": [5, 9, 15, 25, 41], "model__weights": ["uniform", "distance"]},
    "Decision Tree": {"model__max_depth": [6, 8, 12, 16, None], "model__min_samples_leaf": [5, 20, 50, 100]},
    "Neural Network (MLP)": {"model__hidden_layer_sizes": [(64,), (128, 64), (256, 128)],
                             "model__alpha": [1e-4, 1e-3, 1e-2], "model__learning_rate_init": [1e-3, 3e-3]},
}
# Tree ensembles already use all CPU cores, so the search runs their trials one after the other
MULTICORE_MODELS = ["Random Forest", "Extra Trees", "Hist Gradient Boosting", "XGBoost", "LightGBM"]
search = RandomizedSearchCV(models[BEST_NAME], PARAM_GRIDS[BEST_NAME], n_iter=8, scoring="f1_macro",
                            cv=GroupKFold(n_splits=3), n_jobs=1 if BEST_NAME in MULTICORE_MODELS else -1,
                            random_state=RANDOM_STATE, refit=False, verbose=1)
t0 = time.time()
search.fit(X_train, y_train, groups=groups)
print(f"Search done in {time.time() - t0:.0f}s")
print("Best parameters:", search.best_params_)
print(f"CV macro-F1 of the best setting: {search.best_score_:.3f} (3 folds) | default setting: {cv_results.loc[BEST_NAME, 'macro_f1']:.3f} (4 folds)")

In [ ]:
# Keep the tuned version only if it really beats the default on the same 4 folds
tuned = clone(models[BEST_NAME]).set_params(**search.best_params_)
r = cross_validate(tuned, X_train, y_train, groups=groups, cv=cv, scoring="f1_macro")
print(f"4-fold macro-F1: default {cv_results.loc[BEST_NAME, 'macro_f1']:.3f} vs tuned {r['test_score'].mean():.3f}")
if r["test_score"].mean() >= cv_results.loc[BEST_NAME, "macro_f1"]:
    best_model, best_label = tuned, f"{BEST_NAME} (tuned)"
else:
    best_model, best_label = clone(models[BEST_NAME]), f"{BEST_NAME} (default settings)"
print("Final model:", best_label)

## 8. Final evaluation on the unseen 2025 farms

In [ ]:
best_model.fit(X_train, y_train)
pred = best_model.predict(X_test)
print(f"{best_label}")
print(f"Test macro-F1      {f1_score(y_test, pred, average='macro'):.3f}")
print(f"Balanced accuracy  {balanced_accuracy_score(y_test, pred):.3f}")
print(f"Accuracy           {accuracy_score(y_test, pred):.3f}\n")
print(classification_report(y_test, pred, target_names=CLASSES, digits=3))

In [ ]:
cm = confusion_matrix(y_test, pred)
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
fig, ax = plt.subplots(1, 2, figsize=(17, 6))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[0])
ax[0].set(title="Confusion matrix (days)", xlabel="predicted", ylabel="actual")
sns.heatmap(cm_pct, annot=True, fmt=".0f", cmap="Greens", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[1], vmax=100)
ax[1].set(title="Confusion matrix (% of each actual class = recall)", xlabel="predicted", ylabel="actual")
plt.tight_layout(); plt.show()

In [ ]:
# All algorithms on the test farms (each trained on the full training set)
rows = []
for name, model in models.items():
    m = model if name != BEST_NAME else best_model
    if name != BEST_NAME:
        m.fit(X_train, y_train)
    p = m.predict(X_test)
    rows.append({"model": name + (" *" if name == BEST_NAME else ""), "test macro-F1": f1_score(y_test, p, average="macro"),
                 "balanced acc": balanced_accuracy_score(y_test, p), "accuracy": accuracy_score(y_test, p),
                 **{f"F1 {c}": s for c, s in zip(CLASSES, f1_score(y_test, p, average=None, labels=range(len(CLASSES))))}})
test_table = pd.DataFrame(rows).set_index("model").sort_values("test macro-F1", ascending=False)
print("* = selected model")
test_table.round(3)

In [ ]:
# Performance per soil type: does the model work on every soil?
soil_test = test["soil_type"].values
per_soil = pd.DataFrame([{"soil": s, "days": (soil_test == s).sum(),
                          "macro-F1": f1_score(y_test[soil_test == s], pred[soil_test == s], average="macro",
                                               labels=np.unique(y_test[soil_test == s])),
                          "accuracy": accuracy_score(y_test[soil_test == s], pred[soil_test == s])}
                         for s in ["sand", "loamy_sand", "sandy_loam", "loam", "clay_loam", "clay"]]).set_index("soil")
display(per_soil.round(3))

# Which features matter? (drop in test macro-F1 when a feature is shuffled)
idx = np.random.default_rng(RANDOM_STATE).choice(len(X_test), size=min(6000, len(X_test)), replace=False)
pi = permutation_importance(best_model, X_test.iloc[idx], y_test[idx], scoring="f1_macro", n_repeats=3,
                            random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False).head(20)
imp.sort_values().plot.barh(figsize=(8, 7), color="#55A868")
plt.title("Top 20 features (drop in test macro-F1 when shuffled)"); plt.tight_layout(); plt.show()

## 9. Save the model and predict new days

In [ ]:
MODEL_DIR = os.path.join(os.path.dirname(DATA_PATH), "models")
try:
    os.makedirs(MODEL_DIR, exist_ok=True)
    open(os.path.join(MODEL_DIR, ".write_test"), "w").close(); os.remove(os.path.join(MODEL_DIR, ".write_test"))
except OSError:
    MODEL_DIR = "models"; os.makedirs(MODEL_DIR, exist_ok=True)   # no write access to the shared folder

bundle = {"model": best_model, "model_name": best_label, "features": FEATURES, "classes": CLASSES,
          "test_macro_f1": float(f1_score(y_test, pred, average="macro"))}
MODEL_PATH = os.path.join(MODEL_DIR, "stress_model.joblib")
joblib.dump(bundle, MODEL_PATH)
print("Saved:", MODEL_PATH)
print("Note: to load it in another notebook, first define the IQRCapper and BalancedXGBClassifier classes (cells above).")

In [ ]:
def predict_stress(day_rows: pd.DataFrame, bundle: dict) -> pd.DataFrame:
    # Same preparation as training, then class + confidence
    x = encode(add_missing_flags_and_rule_fills(day_rows))
    x = x.reindex(columns=bundle["features"], fill_value=0).astype(float)
    proba = bundle["model"].predict_proba(x)
    out = pd.DataFrame(proba.round(3), columns=[f"p_{c}" for c in bundle["classes"]], index=day_rows.index)
    out.insert(0, "predicted_stress", [bundle["classes"][i] for i in proba.argmax(1)])
    out.insert(1, "confidence", proba.max(1).round(3))
    return out

loaded = joblib.load(MODEL_PATH)
test_days = raw[raw["orchard_id"].str.endswith("Y3")]
sample = pd.concat([g.sample(min(2, len(g)), random_state=1) for _, g in test_days.groupby(TARGET)])   # 2 days per class
result = predict_stress(sample.drop(columns=[TARGET]), loaded)
pd.concat([sample[["tree_id", "soil_type", "date", TARGET]].rename(columns={TARGET: "actual"}), result], axis=1)

## 10. Conclusion
* **Selected algorithm:** see step 6 (cross-validated macro-F1) and step 8 (test on unseen farms). Quote the test macro-F1, the per-class F1 and the per-soil table.
* **Hardest class:** usually `rootzone` (salinity / root damage). None of the sensors measures salinity, so the model relies on irrigation-water EC and on "moist soil but low water uptake". Record the irrigation-water EC for each drip farm (one measurement per season is enough): when it is unknown, the model has less to go on.
* **Soil types:** check the per-soil table. In our run, **clay** was the hardest soil (macro-F1 ≈ 0.48) and loamy sand came next (≈ 0.55). Sand was the easiest (≈ 0.80). In clay, waterlogged days and moist-but-stressed (`rootzone`) days look alike, so collect real labelled days from at least one clay farm.
* This dataset is synthetic (physics-based, calibrated to Tunisian conditions). Before field use, validate the model on real days logged by your nodes, with the farmer's notes and a few reference measurements.